# CEFR Lexical Sophistication Tagger — Notebook

Refactored from [fulan233/cefr-lexical-sophistication](https://github.com/fulan233/cefr-lexical-sophistication)  
Original paper: Hu N, Lu X, Hu R. (2025). *Behavior Research Methods*, 57(8): 226.

**Changes from original:**
- Replaced deprecated `bert-serving` (TF 1.x) with HuggingFace `transformers`
- Works directly with text strings and JSONL files
- Notebook-friendly, no separate server process needed

**You still need the `dict/` folder from the original repo.**

## 0. Setup

In [1]:
# Install dependencies (uncomment if needed)
# !pip install transformers torch nltk numpy pandas tqdm

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)

True

## 1. Clone the repo & check dict files

You need the `dict/` folder from the original repo. If you haven't already:

In [2]:
# Clone the repo (only need to do this once)
# !git clone https://github.com/fulan233/cefr-lexical-sophistication.git

import os

# Set this to wherever you cloned the repo
REPO_DIR = "./"
DICT_DIR = os.path.join(REPO_DIR, "dict")

# Verify the dict files exist
required_files = [
    "EVP_poly_dict.data",
    "EVP_AUG_sense_dict.data", 
    "monodict.data",
    "rule.txt",
    "mono_pos.data",
    "target_level.data",
]
for f in required_files:
    path = os.path.join(DICT_DIR, f)
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"{status}  {f}")

✅  EVP_poly_dict.data
✅  EVP_AUG_sense_dict.data
✅  monodict.data
✅  rule.txt
✅  mono_pos.data
✅  target_level.data


## 2. Initialize the tagger

This loads BERT and all dictionary resources. Takes ~30s on first run (downloads model).

In [3]:
from cefr_tagger import CEFRTagger, cefr_distribution, cefr_feature_vector, mean_sophistication_score

tagger = CEFRTagger(
    dict_dir=DICT_DIR,
    bert_model="bert-base-uncased",  # must match the sense embeddings
    device=None,                      # auto-detect GPU/CPU
    max_seq_len=128,
    batch_size=32,
)

Loaded 2446 polysemous words and 7227 sense embeddings.
Loading BERT model 'bert-base-uncased' on cuda...
BERT model loaded.


## 3. Quick test with a single text

In [5]:
sample_text = """The cat sat on the mat. She was contemplating the 
intricacies of quantum mechanics while sipping her coffee."""

tagged_lines = tagger.tag_text(sample_text)

print("Tagged output:")
for line in tagged_lines:
    print(f"  {line}")

print("\nCEFR distribution:")
dist = cefr_distribution(tagged_lines, lazy_mode="None", target_level=tagger.resources.target_level)
for level, pct in dist.items():
    print(f"  {level}: {pct:.1%}")

print(f"\nMean sophistication score: {mean_sophistication_score(tagged_lines, lazy_mode='None', target_level=tagger.resources.target_level):.2f}")
print(f"Feature vector: {cefr_feature_vector(tagged_lines, lazy_mode='None', target_level=tagger.resources.target_level)}")

Tagged output:
  the-determiner_6#B1 cat_A1 sit-verb_1#A1 on-preposition_1#A1 the-determiner_1#A1 mat_C1 .
  she_A1 be-auxiliaryverb_2#A2 contemplate_C2 the-determiner_5#B2
  int_UN of-preposition_8#B1 quantum_UN mechanic_A2 while-conjunction_1#A2 sip-verb_1#C1 her_A1 coffee_A1 .

CEFR distribution:
  A1: 43.8%
  A2: 18.8%
  B1: 12.5%
  B2: 6.2%
  C1: 12.5%
  C2: 6.2%

Mean sophistication score: 2.17
Feature vector: [0.4375, 0.1875, 0.125, 0.0625, 0.125, 0.0625]


# Test on the inputs provided on GitHub and see if results match
### The difference is only how punctuation is tagged, they had the _UN tag in the output files on GitHub
### But in their code, they added is_alpha in line 85 in tag_text_server (it says the word modified there too)
### which shouldnt have tagged punctuation like this
### Maybe the output files were on a previous version of the code?

In [6]:
## Validation: Compare against original output

import os
from collections import Counter

SAMPLES_DIR = os.path.join(REPO_DIR, "samples")
ORIGINAL_OUTPUT_DIR = os.path.join(REPO_DIR, "output")

summary = Counter()

for filename in sorted(os.listdir(SAMPLES_DIR)):
    if not filename.endswith(".txt"):
        continue
    
    # Read the original input text
    with open(os.path.join(SAMPLES_DIR, filename), 'r', encoding='utf-8') as f:
        text = f.read()
    
    # Read the expected output
    with open(os.path.join(ORIGINAL_OUTPUT_DIR, filename), 'r', encoding='utf-8') as f:
        expected_lines = [line.strip() for line in f if line.strip()]
    
    # Run our tagger
    our_lines = tagger.tag_text(text)
    
    print(f"\n{'='*60}")
    print(f"File: {filename}  (expected {len(expected_lines)} lines, got {len(our_lines)} lines)")
    print(f"{'='*60}")
    
    n_lines = min(len(expected_lines), len(our_lines))
    
    for line_idx in range(n_lines):
        exp_tokens = expected_lines[line_idx].split()
        our_tokens = our_lines[line_idx].split()
        
        n_tokens = max(len(exp_tokens), len(our_tokens))
        
        for tok_idx in range(n_tokens):
            exp = exp_tokens[tok_idx] if tok_idx < len(exp_tokens) else "<MISSING>"
            ours = our_tokens[tok_idx] if tok_idx < len(our_tokens) else "<MISSING>"
            
            if exp == ours:
                summary["exact_match"] += 1
            else:
                # Check if CEFR level matches even if sense ID differs
                exp_level = exp.split("#")[-1] if "#" in exp else exp.split("_")[-1] if "_" in exp else None
                our_level = ours.split("#")[-1] if "#" in ours else ours.split("_")[-1] if "_" in ours else None
                
                if exp_level and our_level and exp_level == our_level:
                    summary["same_level_diff_sense"] += 1
                    print(f"  L{line_idx+1} T{tok_idx+1} LEVEL OK: {exp}  →  {ours}")
                else:
                    summary["disagreement"] += 1
                    print(f"  L{line_idx+1} T{tok_idx+1} MISMATCH: {exp}  →  {ours}")
    
    # Flag line count mismatches
    if len(expected_lines) != len(our_lines):
        summary["line_count_mismatch"] += 1
        print(f"  ⚠️  Line count mismatch: expected {len(expected_lines)}, got {len(our_lines)}")

# Summary
total = summary["exact_match"] + summary["same_level_diff_sense"] + summary["disagreement"]
print(f"\n{'='*60}")
print(f"SUMMARY ({total} tokens compared)")
print(f"{'='*60}")
print(f"  Exact match:              {summary['exact_match']:>5}  ({summary['exact_match']/total:.1%})")
print(f"  Same CEFR, different ID:  {summary['same_level_diff_sense']:>5}  ({summary['same_level_diff_sense']/total:.1%})")
print(f"  Disagreement:             {summary['disagreement']:>5}  ({summary['disagreement']/total:.1%})")
if summary["line_count_mismatch"]:
    print(f"  ⚠️  Files with line count mismatch: {summary['line_count_mismatch']}")


File: 1.txt  (expected 11 lines, got 11 lines)
  L1 T11 MISMATCH: ?_UN  →  ?
  L2 T17 MISMATCH: ._UN  →  .
  L3 T19 MISMATCH: ._UN  →  .
  L4 T2 MISMATCH: ,_UN  →  ,
  L4 T12 MISMATCH: ._UN  →  .
  L5 T19 MISMATCH: ._UN  →  .
  L6 T2 MISMATCH: ,_UN  →  ,
  L6 T10 MISMATCH: ._UN  →  .
  L7 T22 MISMATCH: ._UN  →  .
  L8 T25 MISMATCH: ._UN  →  .
  L9 T27 MISMATCH: ._UN  →  .
  L10 T3 MISMATCH: ,_UN  →  ,
  L10 T19 MISMATCH: ,_UN  →  ,
  L10 T30 MISMATCH: ._UN  →  .
  L11 T17 MISMATCH: ._UN  →  .

File: 2.txt  (expected 27 lines, got 27 lines)
  L1 T9 MISMATCH: ,_UN  →  ,
  L1 T33 MISMATCH: ._UN  →  .
  L2 T12 MISMATCH: ._UN  →  .
  L3 T16 MISMATCH: ._UN  →  .
  L4 T24 MISMATCH: ,_UN  →  ,
  L4 T35 MISMATCH: ,_UN  →  ,
  L4 T50 MISMATCH: ._UN  →  .
  L5 T3 MISMATCH: ,_UN  →  ,
  L6 T10 MISMATCH: ,_UN  →  ,
  L6 T29 MISMATCH: ._UN  →  .
  L7 T23 MISMATCH: ,_UN  →  ,
  L7 T38 MISMATCH: ,_UN  →  ,
  L7 T53 MISMATCH: ._UN  →  .
  L8 T11 MISMATCH: ,_UN  →  ,
  L8 T13 MISMATCH: ,_UN  →  ,
  L8 

# Does this make any difference with the metrics?
### Differences are minimal and highest for _UN tokensm which was expected
### I think it is better to exclude punctuation and keep the is_alpha part

In [7]:
import os

# Save our tagger's output to a separate folder
OUR_OUTPUT_DIR = os.path.join(REPO_DIR, "output_ours")
os.makedirs(OUR_OUTPUT_DIR, exist_ok=True)

for filename in sorted(os.listdir(SAMPLES_DIR)):
    if not filename.endswith(".txt"):
        continue
    with open(os.path.join(SAMPLES_DIR, filename), 'r', encoding='utf-8') as f:
        text = f.read()
    tagged_lines = tagger.tag_text(text)
    with open(os.path.join(OUR_OUTPUT_DIR, filename), 'w', encoding='utf-8') as f:
        for line in tagged_lines:
            f.write(line + '\n')

# export_csv.py uses relative paths for dict/, so we need to chdir
original_dir = os.getcwd()
os.chdir(REPO_DIR)

from export_csv import output_csv

output_csv('./output/*.txt', 'indices_original.csv', tag_mode='None', window_size=100)
output_csv('./output_ours/*.txt', 'indices_ours.csv', tag_mode='None', window_size=100)

os.chdir(original_dir)

# Compare
import pandas as pd

df_orig = pd.read_csv(os.path.join(REPO_DIR, "indices_original.csv")).set_index("filename")
df_ours = pd.read_csv(os.path.join(REPO_DIR, "indices_ours.csv")).set_index("filename")

diff = (df_orig - df_ours).abs()
print("Max absolute difference per index:")
print(diff.max().sort_values(ascending=False).to_string())
print(f"\nOverall max difference: {diff.max().max()}")

Max absolute difference per index:
token_mean_score    0.312
MA_UN_token         0.139
MA_A_token          0.107
type_mean_score     0.093
MA_A1_token         0.079
MA_AboveB1_token    0.048
MA_B_token          0.042
MA_A_type           0.036
MA_A2_token         0.034
MA_AboveB1_type     0.031
MA_B_type           0.026
MA_A1_type          0.024
MA_B1_token         0.024
MA_AboveB2_token    0.023
MA_A2_type          0.018
MA_B2_token         0.017
MA_AboveB2_type     0.016
MA_B1_type          0.015
MA_B2_type          0.011
MA_UN_type          0.010
MA_C_token          0.006
MA_C_type           0.005
MA_C1_type          0.003
MA_C1_token         0.003
MA_C2_token         0.003
MA_C2_type          0.002

Overall max difference: 0.31200000000000006


## 4. Process JSONL dataset

Adapt the field names (`text_field`, `label_field`) to match your JSONL format.

In [8]:
import json
import pandas as pd
from tqdm import tqdm

def load_jsonl(filepath: str) -> list:
    """Load a JSONL file into a list of dicts."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def process_dataset(
    data: list,
    tagger: CEFRTagger,
    text_field: str = "text",
    lazy_mode: str = None,           # None, 'LazyA1', 'Min', 'Rand'
    include_unknown: bool = False,
) -> pd.DataFrame:
    """
    Process a list of dicts, adding CEFR features to each record.
    
    Returns a DataFrame with the original data plus CEFR columns.
    """
    target_level = tagger.resources.target_level  # needed for lazy modes
    
    levels = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    if include_unknown:
        levels.append('UN')
    
    results = []
    for item in tqdm(data, desc="Processing texts"):
        text = item[text_field]
        tagged_lines = tagger.tag_text(text)
        
        dist = cefr_distribution(
            tagged_lines, lazy_mode=lazy_mode,
            target_level=target_level, include_unknown=include_unknown
        )
        score = mean_sophistication_score(
            tagged_lines, lazy_mode=lazy_mode, target_level=target_level
        )
        
        row = dict(item)  # keep original fields
        for level in levels:
            row[f"cefr_{level}"] = dist[level]
        row["cefr_mean_score"] = score
        results.append(row)
    
    return pd.DataFrame(results)

# Try it for the Write and Improve data
In LAZY_MODE
- None = categorises the wod based on the original tagging
- Min = always chooses the lowest CEFR level for polysemous words the span CEFR levels
- Rand = chooses a random CEFR level for polysemous words the span CEFR levels
- LazyA1 = always chooses A1 for words that span CEFR levels but have an A1 sense, everything else matches the original tag

In [10]:
# === Adjust these to your setup ===

TRAIN_PATH = r"C:\Users\coope\Documents\Learner_Writing_Classification\_morphology\Bennett_Lists\_Write_and_Improve\data\train.jsonl"
VAL_PATH = r"C:\Users\coope\Documents\Learner_Writing_Classification\_morphology\Bennett_Lists\_Write_and_Improve\data\val.jsonl"
TEST_PATH = r"C:\Users\coope\Documents\Learner_Writing_Classification\_morphology\Bennett_Lists\_Write_and_Improve\data\test.jsonl"
TEXT_FIELD = "text"        # key in your JSONL that holds the text
LABEL_FIELD = "label"     # key for your classification label
LAZY_MODE = "None"      # or None, 'Min', 'Rand', 'LazyA1'

# Load
train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)
test_data = load_jsonl(TEST_PATH)
print(f"Train: {len(train_data)}, Val: {len(test_data)}, Test: {len(test_data)}")

Train: 4195, Val: 404, Test: 404


In [11]:
# Process (this is the slow step — runs BERT on every text)
train_df = process_dataset(train_data, tagger, text_field=TEXT_FIELD, lazy_mode=LAZY_MODE)
val_df = process_dataset(val_data, tagger, text_field=TEXT_FIELD, lazy_mode=LAZY_MODE)
test_df = process_dataset(test_data, tagger, text_field=TEXT_FIELD, lazy_mode=LAZY_MODE)

train_df.head()

Processing texts: 100%|██████████| 404/404 [00:05<00:00, 73.34it/s]


,id,text,label,topic,prompt_id,learner_id,language,genre,prompt,cefr_A1,cefr_A2,cefr_B1,cefr_B2,cefr_C1,cefr_C2,cefr_mean_score
0,essay_28aba35720851372,New Year is always an invite to establish goal...,B1,A new school year,prompt_e4c497f7613dcf8a,user_e48ee201eb41c367,Portuguese,A report,It is the start of a new school year at your s...,0.429348,0.255435,0.179348,0.081522,0.005435,0.048913,2.015464
1,essay_de0d459ad84b20be,"In addition to studying at university, I have ...",B1,What you do in your free time,prompt_40037060180c75db,user_4024872408b949a3,Vietnamese,An essay,Your teacher asks you to write about what you ...,0.506173,0.345679,0.098765,0.024691,0.012346,0.012346,1.686747
2,essay_39e931900d165aff,"Hi, Cristina, I am feeling not good when I cam...",A2,A school friend,prompt_98e191fa5bded725,user_988d09923a575616,Hindi,An email,"A friend from your class, Cristina, had an acc...",0.562500,0.291667,0.104167,0.020833,0.020833,0.000000,1.549020
3,essay_2932ebd5446803c9,"Hi Adrew, We are going for short holiday next ...",B1,Instructions,prompt_49f744b86a7de069,user_49e44929bd7f9a27,Polish,A note,Your neighbour is going to look after your hou...,0.520270,0.317568,0.121622,0.027027,0.006757,0.006757,1.636364
4,essay_2ae41ea2a3873e11,"HI Ana, Thank you for reaching me out. To answ...",B1,Learning English,prompt_e3b16d9c3c7b3cc8,user_49e44929bd7f9a27,Polish,An email,"Read this email from your cousin, Ana. Hi, I s...",0.465753,0.246575,0.178082,0.095890,0.000000,0.013699,1.743902


In [14]:
# Save to disk so you don't have to re-run BERT every time
# train_df.to_csv("train_with_cefr.csv", index=False)
# test_df.to_csv("test_with_cefr.csv", index=False)

# Or save as JSONL
train_df.to_json("wai_data/train_cefr_AW.jsonl", orient="records", lines=True)
val_df.to_json("wai_data/val_cefr_AW.jsonl", orient="records", lines=True)
test_df.to_json("wai_data/test_cefr_AW.jsonl", orient="records", lines=True)

## 5. Extract feature vectors for your classifier

Once you have the CEFR-enriched DataFrame, extract the feature columns.

In [ ]:
# CEFR feature columns
# cefr_cols = ["cefr_A1", "cefr_A2", "cefr_B1", "cefr_B2", "cefr_C1", "cefr_C2"]

# Example: extract numpy arrays for your model
# train_cefr_features = train_df[cefr_cols].values  # shape: (n_train, 6)
# test_cefr_features = test_df[cefr_cols].values     # shape: (n_test, 6)
# train_labels = train_df[LABEL_FIELD].values

# These can be concatenated with your ModernBERT CLS embeddings, e.g.:
# combined = np.concatenate([modernbert_embeddings, train_cefr_features], axis=1)

## 6. (Optional) Explore the distributions

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

combined_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

cefr_levels = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
cefr_cols = [f'cefr_{level}' for level in cefr_levels]
text_labels = ['A2', 'B1', 'B2', 'C1']

fig1 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'{level} Word Proportion' for level in cefr_levels],
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

for idx, (level, cefr_col) in enumerate(zip(cefr_levels, cefr_cols)):
    row = (idx // 3) + 1
    col = (idx % 3) + 1

    for label in text_labels:
        label_data = combined_df[combined_df[LABEL_FIELD] == label]
        fig1.add_trace(
            go.Box(
                y=label_data[cefr_col],
                name=label,
                marker_color='#2C2C2C',
                line=dict(width=1),
                showlegend=False,
                boxmean=True
            ),
            row=row, col=col
        )

fig1.update_layout(
    template='plotly_white',
    title_text='CEFR Word Proportions by Text Level',
    title_x=0.5,
    height=600,
    showlegend=False
)

for i in range(6):
    fig1.update_yaxes(title_text="Proportion", row=(i // 3) + 1, col=(i % 3) + 1)

for col_idx in range(1, 4):
    fig1.update_xaxes(title_text="Text CEFR Level", row=2, col=col_idx)

fig1.show()

In [13]:
fig2 = go.Figure()

for label in text_labels:
    label_data = combined_df[combined_df[LABEL_FIELD] == label]
    fig2.add_trace(
        go.Box(
            y=label_data['cefr_mean_score'],
            name=label,
            marker_color='#2C2C2C',
            line=dict(width=1),
            boxmean=True
        )
    )

fig2.update_layout(
    template='plotly_white',
    title='CEFR Mean Score Distribution by Text Level',
    title_x=0.5,
    xaxis_title='Text CEFR Level',
    yaxis_title='Mean CEFR Score',
    height=500,
    showlegend=False
)

fig2.show()

## 7. (Optional) Full 50-index computation

If you want the full set of moving-average indices from the original `export_csv.py`,
you can still use it on the tagged output. Just save the tagged lines to files
and run the original script.

In [ ]:
import os

def save_tagged_output(tagged_lines: list, filepath: str):
    """Save tagged lines to a text file (same format as original output/)."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        for line in tagged_lines:
            f.write(line + '\n')

# Example: save all tagged outputs, then use export_csv.py
# output_dir = "./output"
# for i, item in enumerate(train_data):
#     tagged = tagger.tag_text(item[TEXT_FIELD])
#     save_tagged_output(tagged, f"{output_dir}/train_{i:05d}.txt")
#
# Then run: 
# from export_csv import output_csv
# output_csv('./output/train_*.txt', 'train_full_indices.csv', tag_mode='LazyA1', window_size=100)